# 1. Data Collection - SpaceX REST API
Collect Falcon 9 launch data from the SpaceX API and save it as `dataset_part_1.csv`.

In [ ]:
import requests, datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

# Static snapshot of the launches endpoint (so results are reproducible)
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
print('status:', response.status_code)
data = pd.json_normalize(response.json())
data.head()

In [ ]:
# Keep the useful columns, single-core and single-payload launches only
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
data.head()

In [ ]:
BoosterVersion, PayloadMass, Orbit = [], [], []
LaunchSite, Longitude, Latitude = [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            r = requests.get('https://api.spacexdata.com/v4/rockets/' + str(x)).json()
            BoosterVersion.append(r['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            r = requests.get('https://api.spacexdata.com/v4/launchpads/' + str(x)).json()
            Longitude.append(r['longitude'])
            Latitude.append(r['latitude'])
            LaunchSite.append(r['name'])

def getPayloadData(data):
    for load in data['payloads']:
        if load:
            r = requests.get('https://api.spacexdata.com/v4/payloads/' + load).json()
            PayloadMass.append(r['mass_kg'])
            Orbit.append(r['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            r = requests.get('https://api.spacexdata.com/v4/cores/' + core['core']).json()
            Block.append(r['block'])
            ReusedCount.append(r['reuse_count'])
            Serial.append(r['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

In [ ]:
# This makes many API calls and can take a few minutes
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)
print(len(BoosterVersion), len(LaunchSite), len(PayloadMass), len(Outcome))

In [ ]:
launch_dict = {'FlightNumber': list(data['flight_number']), 'Date': list(data['date']),
               'BoosterVersion': BoosterVersion, 'PayloadMass': PayloadMass, 'Orbit': Orbit,
               'LaunchSite': LaunchSite, 'Outcome': Outcome, 'Flights': Flights, 'GridFins': GridFins,
               'Reused': Reused, 'Legs': Legs, 'LandingPad': LandingPad, 'Block': Block,
               'ReusedCount': ReusedCount, 'Serial': Serial, 'Longitude': Longitude, 'Latitude': Latitude}
df = pd.DataFrame(launch_dict)
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print(data_falcon9.shape)
data_falcon9.head()

In [ ]:
# Replace missing payload masses with the mean, then save
mean_payload = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_payload)
print(data_falcon9.isnull().sum())
data_falcon9.to_csv('../data/dataset_part_1.csv', index=False)